# 04 — TGN Training (Kaggle)

Trains a memory-based Temporal Graph Network (TGN) and compares five class imbalance handling strategies. This notebook is designed to run on **Kaggle with a T4 GPU**.

**Input:** `nft_graph_dataset.npz` (upload to Kaggle as a dataset)  
**Output:** `results.json`, `preds_*.npz`

---

### Model
Memory-based TGN using `TGNMemory` with `IdentityMessage` and `LastAggregator`. The neighbor-attention embedding layer is intentionally omitted to keep the model lightweight at this graph scale, making the temporal memory the sole driver of the model's behavior.

### Five imbalance strategies compared

| Strategy | Description |
|----------|-------------|
| Plain BCE | Standard binary cross-entropy (baseline) |
| Weighted BCE | Positive class weighted by `N_neg / N_pos` |
| Class-Balanced | Effective number of samples weighting (β=0.9999) |
| Focal Loss | Down-weights easy examples (α=0.75, γ=2.0) |
| Asymmetric Loss | Stronger focus on easy negatives (γ⁻=4, γ⁺=0, clip=0.05) |

### Config
- `memory_dim = 100`, `time_dim = 100`
- `batch_size = 256`, `lr = 1e-3`, `epochs = 15`
- Seed: 42

## Cell 1 — Setup and config

In [ ]:
!pip install torch_geometric -q

import json, time, gc
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score
)
from torch_geometric.nn.models.tgn import TGNMemory, IdentityMessage, LastAggregator

# Kaggle paths (we used our friend's account to run this on kaggle GPU)
DATA_PATH = "/kaggle/input/datasets/kentnathanael/nft-graph-dataset/nft_graph_dataset.npz"
FOLDER    = "/kaggle/working"

# Config
BATCH_SIZE          = 256
LR                  = 1e-3
EPOCHS              = 15
MEMORY_DIM, TIME_DIM = 100, 100
FOCAL_ALPHA, FOCAL_GAMMA = 0.75, 2.0
CB_BETA             = 0.9999
SEED                = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
assert DEVICE.type == "cuda", "Enable GPU: right panel > Accelerator > GPU T4 x2"

## Cell 2 — Load dataset

In [ ]:
print("Loading dataset...")
data = np.load(DATA_PATH, allow_pickle=True)

src        = torch.tensor(data["src"],        dtype=torch.long)
dst        = torch.tensor(data["dst"],        dtype=torch.long)
timestamps = torch.tensor(data["timestamps"], dtype=torch.long)
edge_feat  = torch.tensor(data["edge_feat"],  dtype=torch.float32)
labels     = torch.tensor(data["labels"],     dtype=torch.float32)
train_mask = torch.tensor(data["train_mask"])
val_mask   = torch.tensor(data["val_mask"])
test_mask  = torch.tensor(data["test_mask"])

num_nodes = int(max(src.max(), dst.max())) + 1
edge_dim  = edge_feat.shape[1]
train_idx = torch.where(train_mask)[0]
val_idx   = torch.where(val_mask)[0]
test_idx  = torch.where(test_mask)[0]

print(f"Nodes: {num_nodes:,}  Edges: {len(src):,}  EdgeDim: {edge_dim}")
print(f"Train: {len(train_idx):,}  Val: {len(val_idx):,}  Test: {len(test_idx):,}")
assert bool(torch.all(timestamps[:-1] <= timestamps[1:])), "Timestamps not sorted!"

## Cell 3 — Model and loss definitions

In [ ]:
class EdgePredictor(nn.Module):
    def __init__(self, memory_dim, edge_feat_dim):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(memory_dim * 2 + edge_feat_dim, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 1))
    def forward(self, z_src, z_dst, edge_feat):
        return self.mlp(torch.cat([z_src, z_dst, edge_feat], dim=1)).squeeze(-1)


class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0):
        super().__init__()
        self.alpha, self.gamma = alpha, gamma
    def forward(self, logits, targets):
        bce = nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        probs = torch.sigmoid(logits)
        pt = torch.where(targets == 1, probs, 1 - probs)
        return (self.alpha * (1 - pt) ** self.gamma * bce).mean()


class AsymmetricLoss(nn.Module):
    def __init__(self, gamma_neg=4.0, gamma_pos=0.0, clip=0.05, eps=1e-8):
        super().__init__()
        self.gamma_neg, self.gamma_pos = gamma_neg, gamma_pos
        self.clip, self.eps = clip, eps
    def forward(self, logits, targets):
        p = torch.sigmoid(logits)
        p_pos = p
        p_neg = 1 - p
        if self.clip and self.clip > 0:
            p_neg = (p_neg + self.clip).clamp(max=1.0)
        los_pos = targets * torch.log(p_pos.clamp(min=self.eps))
        los_neg = (1 - targets) * torch.log(p_neg.clamp(min=self.eps))
        pt = p_pos * targets + p_neg * (1 - targets)
        gamma = self.gamma_pos * targets + self.gamma_neg * (1 - targets)
        return (-(los_pos + los_neg) * (1 - pt) ** gamma).mean()


def build_model():
    memory = TGNMemory(
        num_nodes=num_nodes, raw_msg_dim=edge_dim,
        memory_dim=MEMORY_DIM, time_dim=TIME_DIM,
        message_module=IdentityMessage(edge_dim, MEMORY_DIM, TIME_DIM),
        aggregator_module=LastAggregator()
    ).to(DEVICE)
    predictor = EdgePredictor(MEMORY_DIM, edge_dim).to(DEVICE)
    # Deduplicate shared time_enc parameters to avoid double-update
    params = list(dict.fromkeys(
        list(memory.parameters()) + list(predictor.parameters())
    ))
    optimizer = optim.Adam(params, lr=LR)
    return memory, predictor, optimizer

## Cell 4 — Training and evaluation loops

In [ ]:
def forward_batch(memory, predictor, batch_idx):
    s   = src[batch_idx].to(DEVICE)
    d   = dst[batch_idx].to(DEVICE)
    msg = edge_feat[batch_idx].to(DEVICE)
    n_id = torch.cat([s, d]).unique()
    z, _ = memory(n_id); z = z.detach()  # detach to prevent OOM on large graph
    z_src = z[torch.searchsorted(n_id, s)]
    z_dst = z[torch.searchsorted(n_id, d)]
    return predictor(z_src, z_dst, msg), s, d, msg


def train_one_epoch(memory, predictor, optimizer, criterion):
    memory.train(); predictor.train(); memory.reset_state()
    loader = DataLoader(train_idx.numpy(), batch_size=BATCH_SIZE, shuffle=False)
    total_loss, n = 0.0, 0
    for batch in loader:
        bi = torch.as_tensor(batch, dtype=torch.long)
        t  = timestamps[bi].to(DEVICE)
        optimizer.zero_grad()
        logits, s, d, msg = forward_batch(memory, predictor, bi)
        loss = criterion(logits, labels[bi].to(DEVICE))
        loss.backward(); optimizer.step()
        memory.detach()                   # truncate BPTT between batches
        memory.update_state(s, d, t, msg)
        total_loss += loss.item(); n += 1
        del logits, loss, s, d, msg, t, bi
    return total_loss / max(n, 1)


@torch.no_grad()
def eval_all_splits(memory, predictor):
    """Single chronological pass over all edges, collecting val and test scores."""
    memory.eval(); predictor.eval(); memory.reset_state()
    buckets = {"val": ([], []), "test": ([], [])}
    full = DataLoader(np.arange(len(src)), batch_size=BATCH_SIZE, shuffle=False)
    for step, batch in enumerate(full):
        bi = torch.as_tensor(batch, dtype=torch.long)
        t  = timestamps[bi].to(DEVICE)
        logits, s, d, msg = forward_batch(memory, predictor, bi)
        probs = torch.sigmoid(logits).cpu().numpy().astype(np.float32)
        y     = labels[bi].numpy().astype(np.float32)
        vm    = val_mask[bi].numpy()
        tm    = test_mask[bi].numpy()
        if vm.any():  buckets["val"][0].append(probs[vm]);  buckets["val"][1].append(y[vm])
        if tm.any():  buckets["test"][0].append(probs[tm]); buckets["test"][1].append(y[tm])
        memory.detach()
        memory.update_state(s, d, t, msg)
        del logits, s, d, msg, t, bi
        if step % 50 == 0: torch.cuda.empty_cache()
    out = {k: (np.concatenate(a), np.concatenate(b)) for k, (a, b) in buckets.items()}
    gc.collect(); torch.cuda.empty_cache()
    return out["val"], out["test"]


def best_threshold(scores, y):
    best_t, best_f1 = 0.5, -1.0
    for t in np.linspace(0.05, 0.95, 19):
        pred = (scores >= t).astype(int)
        if pred.sum() == 0: continue
        f1 = f1_score(y, pred, zero_division=0)
        if f1 > best_f1: best_f1, best_t = f1, float(t)
    return best_t, best_f1


def metrics_at(scores, y, thr):
    pred = (scores >= thr).astype(int)
    return {
        "threshold": round(float(thr), 3),
        "precision": round(float(precision_score(y, pred, zero_division=0)), 4),
        "recall":    round(float(recall_score(y, pred, zero_division=0)), 4),
        "f1":        round(float(f1_score(y, pred, zero_division=0)), 4),
        "roc_auc":   round(float(roc_auc_score(y, scores)), 4),
        "pr_auc":    round(float(average_precision_score(y, scores)), 4)
    }


def run_experiment(name, criterion):
    print("\n" + "="*60); print("EXPERIMENT:", name); print("="*60)
    t0 = time.time()
    memory, predictor, optimizer = build_model()
    for epoch in range(1, EPOCHS + 1):
        loss = train_one_epoch(memory, predictor, optimizer, criterion)
        print(f"  epoch {epoch}/{EPOCHS}  loss={loss:.5f}  elapsed={time.time()-t0:.0f}s")
    (val_s, val_y), (test_s, test_y) = eval_all_splits(memory, predictor)
    tuned_t, tuned_val_f1 = best_threshold(val_s, val_y)
    print(f"  tuned threshold (val F1={tuned_val_f1:.4f}): {tuned_t}")
    res = {
        "experiment":    name,
        "epochs":        EPOCHS,
        "test_at_0.5":   metrics_at(test_s, test_y, 0.5),
        "test_at_tuned": metrics_at(test_s, test_y, tuned_t),
        "val_tuned_f1":  round(float(tuned_val_f1), 4),
        "train_minutes": round((time.time() - t0) / 60, 1)
    }
    print("  TEST @0.5  :", res["test_at_0.5"])
    print("  TEST @tuned:", res["test_at_tuned"])
    safe = name.lower().replace(" ", "_")
    np.savez_compressed(f"{FOLDER}/preds_{safe}.npz",
                        val_scores=val_s, val_labels=val_y,
                        test_scores=test_s, test_labels=test_y)
    del memory, predictor, optimizer, val_s, val_y, test_s, test_y
    gc.collect(); torch.cuda.empty_cache()
    return res

## Cell 5 — Run all five experiments

Estimated runtime: ~67 min per strategy, ~335 min total on Kaggle T4.

In [ ]:
def save_all(d):
    with open(f"{FOLDER}/results.json", "w") as f:
        json.dump(d, f, indent=2)

overall_start = time.time()
all_results   = {}
n_pos = float(labels[train_idx].sum())
n_neg = float(len(train_idx) - n_pos)

# 1. Plain BCE (baseline)
all_results["plain_bce"] = run_experiment("Plain BCE", nn.BCEWithLogitsLoss())
save_all(all_results); print(">>> saved plain_bce")

# 2. Weighted BCE
pos_weight = torch.tensor([n_neg / n_pos], device=DEVICE)
all_results["weighted_bce"] = run_experiment(
    "Weighted BCE", nn.BCEWithLogitsLoss(pos_weight=pos_weight))
save_all(all_results); print(">>> saved weighted_bce")

# 3. Class-Balanced Loss (Cui et al., 2019)
eff_pos   = (1.0 - CB_BETA ** n_pos) / (1.0 - CB_BETA)
eff_neg   = (1.0 - CB_BETA ** n_neg) / (1.0 - CB_BETA)
cb_weight = torch.tensor([eff_neg / eff_pos], device=DEVICE)
all_results["class_balanced"] = run_experiment(
    "Class Balanced", nn.BCEWithLogitsLoss(pos_weight=cb_weight))
save_all(all_results); print(">>> saved class_balanced")

# 4. Focal Loss (Lin et al., 2017)
all_results["focal_loss"] = run_experiment(
    "Focal Loss", FocalLoss(FOCAL_ALPHA, FOCAL_GAMMA))
save_all(all_results); print(">>> saved focal_loss")

# 5. Asymmetric Loss
all_results["asymmetric_loss"] = run_experiment(
    "Asymmetric Loss", AsymmetricLoss(gamma_neg=4.0, gamma_pos=0.0, clip=0.05))

# Save final config alongside results
all_results["_config"] = {
    "memory_dim":   MEMORY_DIM,
    "time_dim":     TIME_DIM,
    "batch_size":   BATCH_SIZE,
    "lr":           LR,
    "epochs":       EPOCHS,
    "focal_alpha":  FOCAL_ALPHA,
    "focal_gamma":  FOCAL_GAMMA,
    "cb_beta":      CB_BETA,
    "total_minutes": round((time.time() - overall_start) / 60, 1)
}
save_all(all_results)
print("\nDONE. All 5 experiments saved to /kaggle/working/results.json")